In [ ]:

# ---------------------------------------------------------------------
# CONFIGURAÇÕES INICIAIS DAS ANÁLISES
# ---------------------------------------------------------------------
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F


# Caminhos
SCRIPT_DIR = Path(__file__).resolve().parent
PROJECT_ROOT = Path(__file__).resolve().parents[3]


# Spark
spark = (
    SparkSession.builder
    .appName("TechChallenge_Analytics")
    .master("local[*]")
    .getOrCreate()
)


# ---------------------------------------------------------------------
# PREPARAÇÃO PARA ANALISAR GOLD 06 - Existem diferenças relevantes entre regiões, senioridades ou modelos de trabalho?
# ---------------------------------------------------------------------
PASTA_GOLD = PROJECT_ROOT / "Gold" / "perguntas_negocio"


# Localizar a Gold 06 sem assumir o nome completo da pasta
"""
A pasta é localizada pelo prefixo gold_06 em vez de depender do nome completo do diretório. A validação também interrompe a execução caso nenhuma pasta seja encontrada ou exista mais de uma opção, evitando carregar uma Gold incorreta silenciosamente.
"""
pastas_gold_06 = sorted(
    PASTA_GOLD.glob("gold_06*")
)

print("\nPASTAS GOLD 06 ENCONTRADAS:")

for pasta in pastas_gold_06:
    print(pasta)

if not pastas_gold_06:
    raise FileNotFoundError(
        f"Nenhuma pasta iniciada por 'gold_06' foi encontrada em: {PASTA_GOLD}"
    )

if len(pastas_gold_06) > 1:
    raise RuntimeError(
        "Foi encontrada mais de uma pasta iniciada por 'gold_06'. "
        "Verifique qual deve ser utilizada."
    )

caminho_gold_06 = pastas_gold_06[0]

print("\nGOLD 06 UTILIZADA:")
print(caminho_gold_06)


# Buscar CSVs gerados pelo Spark
arquivos_gold_06 = [
    str(arquivo) for arquivo in caminho_gold_06.glob("part-*.csv")
]

print("\nARQUIVOS ENCONTRADOS:")
print(arquivos_gold_06)

if not arquivos_gold_06:
    raise FileNotFoundError(
        f"Nenhum arquivo part-*.csv encontrado em: {caminho_gold_06}"
    )


# Carregar Gold 06
df_diferencas = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(arquivos_gold_06)
)


# ---------------------------------------------------------------------
# INSPEÇÃO GERAL
# ---------------------------------------------------------------------

"""
A inspeção é realizada antes de definir os recortes analíticos para confirmar como a Gold 06 foi estruturada. O notebook executado identificou 733 linhas e nove colunas, combinando região, senioridade, modelo de trabalho e distribuição por faixa salarial.
"""
print("\n" + "=" * 120)
print("1. INSPEÇÃO GERAL DA GOLD 06")
print("=" * 120)

print("\nAMOSTRA:")
df_diferencas.show(30, truncate=False)

print("\nSCHEMA:")
df_diferencas.printSchema()

print("\nQUANTIDADE DE LINHAS:")
print(df_diferencas.count())

print("\nCOLUNAS:")
print(df_diferencas.columns)


# ---------------------------------------------------------------------
# EDIÇÕES DISPONÍVEIS
# ---------------------------------------------------------------------

"""
A Gold 06 contém apenas as edições 2024-2025 e 2025-2026. Portanto, qualquer evolução histórica construída a partir desta base deverá considerar duas edições, sem assumir disponibilidade de dados para 2023-2024.
"""
print("\n" + "=" * 120)
print("2. EDIÇÕES DISPONÍVEIS")
print("=" * 120)

if "edicao" in df_diferencas.columns:
    (
        df_diferencas
        .select("edicao")
        .distinct()
        .orderBy("edicao")
        .show(20, truncate=False)
    )

else:
    print("A coluna 'edicao' não existe na Gold 06.")


# ---------------------------------------------------------------------
# QUANTIDADE DE VALORES DISTINTOS POR COLUNA
# ---------------------------------------------------------------------

"""
A contagem de valores distintos ajuda a entender a granularidade disponível antes dos cruzamentos. Os outputs confirmam cinco regiões, quatro níveis de senioridade, quatro modelos de trabalho e 13 faixas salariais.
"""
print("\n" + "=" * 120)
print("3. QUANTIDADE DE VALORES DISTINTOS POR COLUNA")
print("=" * 120)

for coluna in df_diferencas.columns:
    qtd_distintos = (
        df_diferencas
        .select(coluna)
        .distinct()
        .count()
    )

    print(f"{coluna}: {qtd_distintos}")


# ---------------------------------------------------------------------
# VALORES DAS COLUNAS CATEGÓRICAS
# ---------------------------------------------------------------------

"""
A inspeção dos valores categóricos é necessária para validar quais grupos podem ser comparados sem criar classificações artificiais. A base já apresenta cinco regiões brasileiras, quatro níveis de senioridade e quatro modalidades de trabalho claramente definidas.
"""
print("\n" + "=" * 120)
print("4. VALORES DAS COLUNAS CATEGÓRICAS")
print("=" * 120)

for coluna in df_diferencas.columns:
    qtd_distintos = (
        df_diferencas
        .select(coluna)
        .distinct()
        .count()
    )

    if qtd_distintos <= 30:
        print("\n" + "-" * 100)
        print(f"COLUNA: {coluna}")
        print("-" * 100)

        (
            df_diferencas
            .select(coluna)
            .distinct()
            .orderBy(coluna)
            .show(100, truncate=False)
        )


# ---------------------------------------------------------------------
# NULOS POR COLUNA
# ---------------------------------------------------------------------

"""
A validação não encontrou valores nulos em nenhuma das nove colunas. Assim, a continuidade da análise não exige exclusão ou imputação de registros por ausência de região, nível, modelo de trabalho ou faixa salarial.
"""
print("\n" + "=" * 120)
print("5. NULOS POR COLUNA")
print("=" * 120)

df_nulos = (
    df_diferencas
    .select(
        [
            F.sum(
                F.when(F.col(coluna).isNull(), 1).otherwise(0)
            ).alias(coluna)

            for coluna in df_diferencas.columns
        ]
    )
)

df_nulos.show(truncate=False)


# ---------------------------------------------------------------------
# POSSÍVEIS COLUNAS RELACIONADAS À PERGUNTA 6
# ---------------------------------------------------------------------

"""
A identificação automática procura no schema os campos relacionados diretamente à pergunta de negócio. O resultado encontrou regiao_onde_mora, nivel e modelo_de_trabalho_atual.

A remuneração não aparece como uma coluna própria: o notebook confirma que a Gold utiliza variavel = "faixa_salarial" e armazena cada intervalo salarial na coluna valor. Essa estrutura deve ser considerada nas análises posteriores de diferença salarial.
"""
print("\n" + "=" * 120)
print("6. POSSÍVEIS COLUNAS PARA REGIÃO, SENIORIDADE E MODELO DE TRABALHO")
print("=" * 120)

termos_interesse = [
    "regiao",
    "região",
    "estado",
    "uf",
    "local",
    "senior",
    "nivel",
    "nível",
    "trabalho",
    "modelo",
    "remoto",
    "presencial",
    "salario",
    "salário",
    "remuneracao",
    "remuneração"
]

colunas_interesse = [
    coluna
    for coluna in df_diferencas.columns
    if any(
        termo.lower() in coluna.lower()
        for termo in termos_interesse
    )
]

print("\nCOLUNAS IDENTIFICADAS:")

for coluna in colunas_interesse:
    print(coluna)


# ---------------------------------------------------------------------
# AMOSTRA SOMENTE DAS COLUNAS DE INTERESSE
# ---------------------------------------------------------------------

"""
A amostra final confirma que região, senioridade e modelo de trabalho coexistem no mesmo registro. Isso permite que as próximas etapas avaliem diferenças salariais considerando esses recortes de forma isolada ou combinada, respeitando o tamanho das amostras de cada grupo.
"""
print("\n" + "=" * 120)
print("7. AMOSTRA DAS COLUNAS DE INTERESSE")
print("=" * 120)

if colunas_interesse:
    (
        df_diferencas
        .select(*colunas_interesse)
        .show(50, truncate=False)
    )

else:
    print(
        "Nenhuma coluna foi identificada automaticamente. "
        "Vamos utilizar a lista completa de colunas da etapa 1."
    )